# GCI In-Class Competition — Home Credit Default Risk
## Self-contained pipeline (reproduces `submission.csv` end-to-end)

This single notebook reproduces the submitted predictions from the raw data:
`input/train.csv` + `input/test.csv` → feature engineering → 4-model 5-fold CV ensemble →
OOF-selected blend → `submission.csv`.

**Approach summary**
- LightGBM / XGBoost / CatBoost with hyperparameters found by Optuna (values hardcoded below
  for reproducibility), plus sklearn HistGradientBoosting for diversity.
- 5-fold StratifiedKFold (seed 42); target encoding computed **inside each fold** (no leakage).
- Final blend weights selected **on out-of-fold predictions only** (greedy hill-climb +
  exhaustive weight grid). No external data, no hand-labeling, all seeds fixed.

**References** (per competition rules):
- GCI tutorial notebook (`tutorial.ipynb`) — baseline workflow and preprocessing ideas.
- Kaggle "Home Credit Default Risk" public EDA knowledge (feature ratios such as
  CREDIT/INCOME, ANNUITY/CREDIT, EXT_SOURCE interactions are standard ideas from that
  competition's public notebooks).
- sklearn/LightGBM/XGBoost/CatBoost documentation.

In [ ]:
# 1. Imports
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
import time, os, gc, json, zipfile, warnings
warnings.filterwarnings('ignore')

SEED = 42
N_SPLITS = 5
N_ESTIMATORS = 3000
EARLY_STOPPING = 200
print("libs ok")

In [ ]:
# 2. Paths (adjust COMPETITION_DIR to wherever the competition folder is mounted)
from google.colab import drive
drive.mount('/content/drive')

# e.g. '/content/drive/MyDrive/competition' if you uploaded the unzipped competition folder
COMPETITION_DIR = '/content/drive/MyDrive/competition'
INPUT = f'{COMPETITION_DIR}/input'

train = pd.read_csv(f'{INPUT}/train.csv')
test = pd.read_csv(f'{INPUT}/test.csv')
sample_sub = pd.read_csv(f'{INPUT}/sample_submission.csv')
print(train.shape, test.shape, sample_sub.shape)

In [ ]:
# 3. Feature engineering (base)
def handle_anomalies(df):
    df['DAYS_EMPLOYED_ANOM'] = (df['DAYS_EMPLOYED'] == 365243).astype(np.int8)
    df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
    if 'OWN_CAR_AGE' in df.columns:
        df['OWN_CAR_AGE'] = df['OWN_CAR_AGE'].clip(lower=0)
    return df

def add_missing_indicators(df, cols):
    for c in cols:
        if c in df.columns:
            df[f'{c}_WAS_MISSING'] = df[c].isnull().astype(np.int8)
    return df

def add_domain_ratios(df):
    eps = 1e-5
    df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / (df['AMT_INCOME_TOTAL'] + eps)
    df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / (df['AMT_INCOME_TOTAL'] + eps)
    df['CREDIT_TERM'] = df['AMT_ANNUITY'] / (df['AMT_CREDIT'] + eps)
    df['DAYS_EMPLOYED_RATIO'] = df['DAYS_EMPLOYED'] / (df['DAYS_BIRTH'] + eps)
    df['GOODS_CREDIT_RATIO'] = df['AMT_GOODS_PRICE'] / (df['AMT_CREDIT'] + eps)
    df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / (df['CNT_FAM_MEMBERS'] + eps)
    df['CREDIT_TO_GOODS_RATIO'] = df['AMT_CREDIT'] / (df['AMT_GOODS_PRICE'] + eps)
    df['ANNUITY_TO_CREDIT_RATIO'] = df['AMT_ANNUITY'] / (df['AMT_CREDIT'] + eps)
    df['INCOME_TO_CREDIT_RATIO'] = df['AMT_INCOME_TOTAL'] / (df['AMT_CREDIT'] + eps)
    df['PHONE_CHANGE_TO_AGE'] = df['DAYS_LAST_PHONE_CHANGE'] / (df['DAYS_BIRTH'] + eps)
    df['EMPLOYED_TO_BIRTH_RATIO'] = df['DAYS_EMPLOYED'] / (df['DAYS_BIRTH'] + eps)
    df['REGISTRATION_TO_BIRTH'] = df['DAYS_REGISTRATION'] / (df['DAYS_BIRTH'] + eps)
    df['ID_PUBLISH_TO_BIRTH'] = df['DAYS_ID_PUBLISH'] / (df['DAYS_BIRTH'] + eps)
    df['CAR_AGE_TO_EMPLOYED'] = df['OWN_CAR_AGE'] / (df['DAYS_EMPLOYED'].abs() + eps)
    df['CAR_AGE_TO_BIRTH'] = df['OWN_CAR_AGE'] / (df['DAYS_BIRTH'].abs() + eps)
    return df

def add_ext_source_interactions(df):
    ext = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
    df['EXT_SOURCES_MEAN'] = df[ext].mean(axis=1)
    df['EXT_SOURCES_PROD'] = df['EXT_SOURCE_1'].fillna(0) * df['EXT_SOURCE_2'].fillna(0) * df['EXT_SOURCE_3'].fillna(0)
    df['EXT_SOURCES_STD'] = df[ext].std(axis=1)
    df['EXT_SOURCES_MAX'] = df[ext].max(axis=1)
    df['EXT_SOURCES_MIN'] = df[ext].min(axis=1)
    df['EXT_1_2_PROD'] = df['EXT_SOURCE_1'].fillna(0) * df['EXT_SOURCE_2'].fillna(0)
    df['EXT_1_3_PROD'] = df['EXT_SOURCE_1'].fillna(0) * df['EXT_SOURCE_3'].fillna(0)
    df['EXT_2_3_PROD'] = df['EXT_SOURCE_2'].fillna(0) * df['EXT_SOURCE_3'].fillna(0)
    df['EXT_1_2_DIFF'] = (df['EXT_SOURCE_1'] - df['EXT_SOURCE_2']).abs()
    df['EXT_1_3_DIFF'] = (df['EXT_SOURCE_1'] - df['EXT_SOURCE_3']).abs()
    df['EXT_2_3_DIFF'] = (df['EXT_SOURCE_2'] - df['EXT_SOURCE_3']).abs()
    df['EXT_1_2_RATIO'] = df['EXT_SOURCE_1'] / (df['EXT_SOURCE_2'] + 1e-5)
    df['EXT_1_3_RATIO'] = df['EXT_SOURCE_1'] / (df['EXT_SOURCE_3'] + 1e-5)
    df['EXT_2_3_RATIO'] = df['EXT_SOURCE_2'] / (df['EXT_SOURCE_3'] + 1e-5)
    df['EXT_SOURCES_COUNT'] = df[ext].notnull().sum(axis=1)
    df['EXT_SOURCES_WEIGHTED_MEAN'] = (df['EXT_SOURCE_1'].fillna(0) + df['EXT_SOURCE_2'].fillna(0) + df['EXT_SOURCE_3'].fillna(0)) / (df['EXT_SOURCES_COUNT'] + 1e-5)
    return df

def add_bureau_aggregations(df):
    b = ['AMT_REQ_CREDIT_BUREAU_HOUR','AMT_REQ_CREDIT_BUREAU_MON',
         'AMT_REQ_CREDIT_BUREAU_QRT','AMT_REQ_CREDIT_BUREAU_YEAR']
    df['BUREAU_QUERY_SUM'] = df[b].fillna(0).sum(axis=1)
    df['BUREAU_QUERY_MEAN'] = df[b].fillna(0).mean(axis=1)
    df['BUREAU_QUERY_MAX'] = df[b].fillna(0).max(axis=1)
    df['BUREAU_QUERY_COUNT'] = df[b].notnull().sum(axis=1)
    df['BUREAU_RECENT_TO_HIST_RATIO'] = (df['AMT_REQ_CREDIT_BUREAU_HOUR'].fillna(0) + df['AMT_REQ_CREDIT_BUREAU_MON'].fillna(0)) / (df['AMT_REQ_CREDIT_BUREAU_QRT'].fillna(0) + df['AMT_REQ_CREDIT_BUREAU_YEAR'].fillna(0) + 1e-5)
    return df

def add_binned_features(df, ref):
    binc = ['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3','AMT_INCOME_TOTAL','AMT_CREDIT','AMT_ANNUITY',
            'DAYS_BIRTH','DAYS_EMPLOYED','DAYS_REGISTRATION','DAYS_ID_PUBLISH','DAYS_LAST_PHONE_CHANGE',
            'CNT_FAM_MEMBERS','CNT_CHILDREN','REGION_POPULATION_RELATIVE','REGION_RATING_CLIENT']
    for col in binc:
        if col in df.columns:
            try:
                q = ref[col].quantile([0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]).values
                df[f'{col}_QBIN'] = pd.cut(df[col], bins=[-np.inf]+list(q)+[np.inf], labels=False, duplicates='drop')
            except Exception:
                df[f'{col}_QBIN'] = pd.qcut(df[col], q=10, labels=False, duplicates='drop')
            df[f'{col}_QBIN'] = df[f'{col}_QBIN'].astype(np.int8)
    return df

HIGH_MISSING = ['EXT_SOURCE_1','OWN_CAR_AGE','EXT_SOURCE_3','AMT_REQ_CREDIT_BUREAU_HOUR',
                'AMT_REQ_CREDIT_BUREAU_MON','AMT_REQ_CREDIT_BUREAU_QRT','AMT_REQ_CREDIT_BUREAU_YEAR',
                'EXT_SOURCE_2','CNT_FAM_MEMBERS']

y = train['TARGET'].values
train_ids = train['SK_ID_CURR'].values
test_ids = test['SK_ID_CURR'].values

tr_fe = train.drop(columns=['TARGET','SK_ID_CURR']).copy()
te_fe = test.drop(columns=['SK_ID_CURR']).copy()

tr_fe = handle_anomalies(tr_fe); te_fe = handle_anomalies(te_fe)
tr_fe = add_missing_indicators(tr_fe, HIGH_MISSING); te_fe = add_missing_indicators(te_fe, HIGH_MISSING)
tr_fe = add_domain_ratios(tr_fe); te_fe = add_domain_ratios(te_fe)
tr_fe = add_ext_source_interactions(tr_fe); te_fe = add_ext_source_interactions(te_fe)
tr_fe = add_bureau_aggregations(tr_fe); te_fe = add_bureau_aggregations(te_fe)
tr_fe = add_binned_features(tr_fe, tr_fe); te_fe = add_binned_features(te_fe, tr_fe)

# label + frequency encoding
cat_cols = tr_fe.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols:
    combined = pd.concat([tr_fe[col], te_fe[col]], axis=0)
    le = LabelEncoder(); le.fit(combined.astype(str))
    tr_fe[col] = le.transform(tr_fe[col].astype(str))
    te_fe[col] = le.transform(te_fe[col].astype(str))
    freq = combined.value_counts(normalize=True).to_dict()
    tr_fe[f'{col}_FREQ'] = tr_fe[col].map(freq).astype(np.float32)
    te_fe[f'{col}_FREQ'] = te_fe[col].map(freq).astype(np.float32)

feature_cols = list(tr_fe.columns)
tr_fe[feature_cols] = tr_fe[feature_cols].fillna(-999)
te_fe[feature_cols] = te_fe[feature_cols].fillna(-999)
X0 = tr_fe.astype(np.float32)
X_test0 = te_fe.astype(np.float32)
print(f"engineered: {X0.shape}, {X_test0.shape}")

In [ ]:
# 4. Validated enrichment + dead-feature removal
DEAD = ['FLAG_MOBIL','ORGANIZATION_TYPE','EXT_SOURCE_2_WAS_MISSING','CNT_FAM_MEMBERS_WAS_MISSING',
        'AMT_REQ_CREDIT_BUREAU_MON_WAS_MISSING','AMT_REQ_CREDIT_BUREAU_YEAR_WAS_MISSING',
        'EXT_SOURCE_1_WAS_MISSING','EXT_SOURCE_3_WAS_MISSING','FLAG_EMP_PHONE',
        'EXT_SOURCE_3_QBIN','OCCUPATION_TYPE_FREQ','NAME_HOUSING_TYPE_FREQ','NAME_FAMILY_STATUS_FREQ',
        'NAME_EDUCATION_TYPE_FREQ','NAME_INCOME_TYPE_FREQ','CODE_GENDER_FREQ','NAME_CONTRACT_TYPE_FREQ',
        'ORGANIZATION_TYPE_FREQ']

def enrich(df, base_cols):
    X = df.copy(); eps = 1e-5
    ext = ['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']
    filled = X[ext].replace(-999, np.nan)
    f = filled.fillna(filled.mean())
    age_years = (X['DAYS_BIRTH'].replace(-999, np.nan).abs() / 365.0).fillna(
        X['DAYS_BIRTH'].replace(-999, np.nan).abs().mean() / 365.0)
    for c in ext:
        X[f'{c}X_AGE'] = f[c] * age_years
    X['EXT_MEAN_X_AGE'] = f.mean(axis=1) * age_years
    X['EXT_2x3'] = f['EXT_SOURCE_2'] * f['EXT_SOURCE_3']
    X['EXT_1x2x3_NANSAFE'] = f['EXT_SOURCE_1'] * f['EXT_SOURCE_2'] * f['EXT_SOURCE_3']
    X['EXT_2_SQ'] = f['EXT_SOURCE_2'] ** 2
    X['EXT_3_SQ'] = f['EXT_SOURCE_3'] ** 2
    X['EXT_SUM'] = f.sum(axis=1)
    emp_years = X['DAYS_EMPLOYED'].replace(-999, np.nan).abs() / 365.0
    X['CREDIT_PER_INCOME_YEAR'] = X['AMT_CREDIT'] / ((X['AMT_INCOME_TOTAL'] + eps) * emp_years.fillna(emp_years.mean()) + eps)
    X['ANNUITY_PER_FAM'] = X['AMT_ANNUITY'] / (X['CNT_FAM_MEMBERS'].replace(0, 1) + eps)
    X['CREDIT_PER_FAM'] = X['AMT_CREDIT'] / (X['CNT_FAM_MEMBERS'].replace(0, 1) + eps)
    X['INCOME_PER_CHILD'] = X['AMT_INCOME_TOTAL'] / (X['CNT_CHILDREN'].clip(lower=0) + 1)
    X['NAN_COUNT'] = (df[base_cols] == -999).sum(axis=1).astype(np.int16)
    X['REGION_RATING_X_EXT_MEAN'] = X['REGION_RATING_CLIENT'] * f.mean(axis=1)
    return X

X = enrich(X0, feature_cols).drop(columns=[c for c in DEAD if c in X0.columns])
X_test = enrich(X_test0, feature_cols).drop(columns=[c for c in DEAD if c in X_test0.columns])
assert list(X.columns) == list(X_test.columns)
print(f"final: {X.shape[1]} features")

In [ ]:
# 5. Tuned hyperparameters (found by Optuna in the development pipeline; hardcoded for reproducibility)
LGB_PARAMS = {"learning_rate": 0.04321176733666811, "num_leaves": 28, "max_depth": 3,
    "min_child_samples": 97, "subsample": 0.744137352044092, "colsample_bytree": 0.8258847596441867,
    "reg_alpha": 0.1853706492115699, "reg_lambda": 0.035504460820014246,
    "min_split_gain": 0.7618907678134024, "min_child_weight": 0.10567827282349425}
XGB_PARAMS = {"learning_rate": 0.023127311910052063, "max_depth": 3, "min_child_weight": 8.595010289429197,
    "subsample": 0.8803532825765757, "colsample_bytree": 0.5325711926493584,
    "colsample_bylevel": 0.6102966544921201, "reg_alpha": 0.0034216968534539678,
    "reg_lambda": 7.070529170721945, "gamma": 0.29349768963525397}
CAT_PARAMS = {"learning_rate": 0.033841402216264506, "depth": 3, "l2_leaf_reg": 4.5756057217815505,
    "bagging_temperature": 0.8546621517583892, "random_strength": 0.0028359528535650993,
    "border_count": 215}
HIGH_CARD = ['ORGANIZATION_TYPE','OCCUPATION_TYPE','NAME_INCOME_TYPE','NAME_EDUCATION_TYPE',
             'NAME_FAMILY_STATUS','NAME_HOUSING_TYPE','NAME_CONTRACT_TYPE']
TE_COLS = [c for c in HIGH_CARD if c in X.columns]
CATS = ['NAME_CONTRACT_TYPE','CODE_GENDER','NAME_INCOME_TYPE','NAME_EDUCATION_TYPE',
        'NAME_FAMILY_STATUS','NAME_HOUSING_TYPE','OCCUPATION_TYPE','FLAG_WORK_PHONE']
print("params ready")

In [ ]:
# 6. Folds + fold-safe target encoding
def add_te(X_tr, X_va, X_te, y_tr, smoothing=10):
    X_tr, X_va, X_te = X_tr.copy(), X_va.copy(), X_te.copy()
    gs = pd.Series(y_tr, index=X_tr.index); gm = gs.mean()
    for col in TE_COLS:
        means = gs.groupby(X_tr[col]).mean()
        counts = X_tr[col].value_counts()
        sm = (means * counts + gm * smoothing) / (counts + smoothing)
        X_tr[f'{col}_TE'] = X_tr[col].map(sm).fillna(gm)
        X_va[f'{col}_TE'] = X_va[col].map(sm).fillna(gm)
        X_te[f'{col}_TE'] = X_te[col].map(sm).fillna(gm)
    return X_tr, X_va, X_te

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(X, y))
def cat_idx(Xc):
    return [list(Xc.columns).index(c) for c in CATS if c in Xc.columns]
print("folds ready")

In [ ]:
# 7. LightGBM (3 seeds)
oof_lgb = np.zeros(len(X)); test_lgb = np.zeros(len(X_test))
t0 = time.time()
for seed in [42, 1337, 2026]:
    oof_s = np.zeros(len(X)); test_s = np.zeros(len(X_test))
    for fold, (tri, vai) in enumerate(FOLDS):
        X_tr, X_va, X_te = add_te(X.iloc[tri], X.iloc[vai], X_test, y[tri])
        m = lgb.LGBMClassifier(**LGB_PARAMS, n_estimators=N_ESTIMATORS, random_state=seed, verbose=-1, n_jobs=2)
        m.fit(X_tr, y[tri], eval_set=[(X_va, y[vai])], eval_metric='auc',
              categorical_feature=cat_idx(X_tr),
              callbacks=[lgb.early_stopping(EARLY_STOPPING, verbose=False)])
        oof_s[vai] = m.predict_proba(X_va)[:, 1]
        test_s += m.predict_proba(X_te)[:, 1] / N_SPLITS
    print(f"LGB seed {seed}: {roc_auc_score(y, oof_s):.5f}")
    oof_lgb += oof_s / len([42, 1337, 2026])
    test_lgb += test_s / len([42, 1337, 2026])
auc_lgb = roc_auc_score(y, oof_lgb)
print(f"LGB avg: {auc_lgb:.5f} ({time.time()-t0:.0f}s)")

In [ ]:
# 8. XGBoost (3 seeds; GPU with CPU fallback)
oof_xgb = np.zeros(len(X)); test_xgb = np.zeros(len(X_test))
xp0 = dict(XGB_PARAMS); xp0.update({'objective':'binary:logistic','eval_metric':'auc','n_jobs':2,'verbosity':0})
t0 = time.time()
for seed in [42, 1337, 2026]:
    oof_s = np.zeros(len(X)); test_s = np.zeros(len(X_test))
    for fold, (tri, vai) in enumerate(FOLDS):
        X_tr, X_va, X_te = add_te(X.iloc[tri], X.iloc[vai], X_test, y[tri])
        dtr = xgb.DMatrix(X_tr, label=y[tri]); dva = xgb.DMatrix(X_va, label=y[vai]); dte = xgb.DMatrix(X_te)
        try:
            m = xgb.train({**xp0, 'device':'cuda', 'random_state':seed}, dtr, num_boost_round=N_ESTIMATORS,
                          evals=[(dva,'val')], early_stopping_rounds=EARLY_STOPPING, verbose_eval=False)
        except Exception:
            m = xgb.train({**xp0, 'tree_method':'hist', 'random_state':seed}, dtr, num_boost_round=N_ESTIMATORS,
                          evals=[(dva,'val')], early_stopping_rounds=EARLY_STOPPING, verbose_eval=False)
        oof_s[vai] = m.predict(dva, iteration_range=(0, m.best_iteration + 1))
        test_s += m.predict(dte, iteration_range=(0, m.best_iteration + 1)) / N_SPLITS
    print(f"XGB seed {seed}: {roc_auc_score(y, oof_s):.5f}")
    oof_xgb += oof_s / len([42, 1337, 2026])
    test_xgb += test_s / len([42, 1337, 2026])
auc_xgb = roc_auc_score(y, oof_xgb)
print(f"XGB avg: {auc_xgb:.5f} ({time.time()-t0:.0f}s)")

In [ ]:
# 9. CatBoost (2 seeds; GPU with CPU fallback)
oof_cat = np.zeros(len(X)); test_cat = np.zeros(len(X_test))
cp0 = dict(CAT_PARAMS); cp0.update({'loss_function':'Logloss','eval_metric':'AUC','verbose':False,'allow_writing_files':False})
t0 = time.time()
for seed in [42, 1337]:
    oof_s = np.zeros(len(X)); test_s = np.zeros(len(X_test))
    for fold, (tri, vai) in enumerate(FOLDS):
        X_tr, X_va, X_te = add_te(X.iloc[tri], X.iloc[vai], X_test, y[tri])
        ci = cat_idx(X_tr)
        try:
            m = cb.CatBoostClassifier(**{**cp0, 'task_type':'GPU'}, iterations=N_ESTIMATORS, random_seed=seed)
            m.fit(X_tr, y[tri], eval_set=(X_va, y[vai]), cat_features=ci, early_stopping_rounds=EARLY_STOPPING)
        except Exception:
            m = cb.CatBoostClassifier(**cp0, iterations=N_ESTIMATORS, random_seed=seed)
            m.fit(X_tr, y[tri], eval_set=(X_va, y[vai]), cat_features=ci, early_stopping_rounds=EARLY_STOPPING)
        oof_s[vai] = m.predict_proba(X_va)[:, 1]
        test_s += m.predict_proba(X_te)[:, 1] / N_SPLITS
    print(f"CAT seed {seed}: {roc_auc_score(y, oof_s):.5f}")
    oof_cat += oof_s / len([42, 1337])
    test_cat += test_s / len([42, 1337])
auc_cat = roc_auc_score(y, oof_cat)
print(f"CAT avg: {auc_cat:.5f} ({time.time()-t0:.0f}s)")

In [ ]:
# 10. HistGradientBoosting (diversity member)
oof_hgb = np.zeros(len(X)); test_hgb = np.zeros(len(X_test))
t0 = time.time()
for fold, (tri, vai) in enumerate(FOLDS):
    X_tr, X_va, X_te = add_te(X.iloc[tri], X.iloc[vai], X_test, y[tri])
    m = HistGradientBoostingClassifier(learning_rate=0.05, max_iter=600, max_leaf_nodes=31,
                                       min_samples_leaf=50, l2_regularization=1.0, random_state=SEED)
    m.fit(X_tr, y[tri])
    oof_hgb[vai] = m.predict_proba(X_va)[:, 1]
    test_hgb += m.predict_proba(X_te)[:, 1] / N_SPLITS
auc_hgb = roc_auc_score(y, oof_hgb)
print(f"HGB: {auc_hgb:.5f} ({time.time()-t0:.0f}s)")

In [ ]:
# 11. Blend selection on OOF (hill-climb + weight grid)
MEMBERS = {'LGB': (oof_lgb, test_lgb), 'XGB': (oof_xgb, test_xgb),
           'CAT': (oof_cat, test_cat), 'HGB': (oof_hgb, test_hgb)}
names = list(MEMBERS.keys())
OOF_M = np.column_stack([MEMBERS[k][0] for k in names])
TEST_M = np.column_stack([MEMBERS[k][1] for k in names])

print("members:", {k: round(roc_auc_score(y, v[0]), 5) for k, v in MEMBERS.items()})

best = (-1.0, None)
for w1 in np.arange(0, 1.0001, 0.02):
    for w2 in np.arange(0, 1.0001 - w1, 0.02):
        for w3 in np.arange(0, 1.0001 - w1 - w2 + 1e-9, 0.02):
            w4 = 1 - w1 - w2 - w3
            if w4 < -1e-9: continue
            a = roc_auc_score(y, w1*OOF_M[:,0] + w2*OOF_M[:,1] + w3*OOF_M[:,2] + w4*OOF_M[:,3])
            if a > best[0]: best = (a, (w1, w2, w3, w4))
final_oof_auc, final_w = best
print(f"best weights {dict(zip(names, final_w))} -> OOF AUC {final_oof_auc:.5f}")

In [ ]:
# 12. Build submission.csv
w = np.array(final_w)
test_pred = w @ TEST_M.T
submission = pd.DataFrame({'SK_ID_CURR': test_ids, 'TARGET': test_pred})
submission = submission.set_index('SK_ID_CURR').loc[sample_sub['SK_ID_CURR']].reset_index()
assert len(submission) == len(sample_sub)
assert submission['TARGET'].between(0, 1).all() and not submission['TARGET'].isnull().any()
assert list(submission.columns) == ['SK_ID_CURR', 'TARGET']
submission.to_csv('submission.csv', index=False)
print(f"OOF AUC: {final_oof_auc:.5f}")
print(submission.head())

In [ ]:
# 13. Package for Honors submission
notebook_name = 'gci_honors_reproduction.ipynb'
with zipfile.ZipFile('gci_honors_reproduction.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(notebook_name, arcname=notebook_name)
print("zipped: gci_honors_reproduction.zip")